In [1]:
import numpy as np
import scipy as sp
import matplotlib.pyplot as plt
import cvxpy as cp
from scipy.special import eval_hermitenorm
from math import factorial

In [2]:
from time_series.data_handlers import TimeSeriesData
from time_series.models import EigenRascuttiModel, KernelRidgeRegression

2026-03-16 14:52:01.265 | INFO     | time_series.config:<module>:13 - PROJ_ROOT path is: /home/james/Repo/PhD Repo/time_series_clustering


In [3]:
def create_dataset(theta, n_points, n_correlated_dimensions, n_uncorrelated_dimensions, noise = 0):
    n_dim = n_correlated_dimensions + n_uncorrelated_dimensions

    # Generate transition matrix
    T = np.zeros((n_dim, n_dim))
    np.fill_diagonal(T, np.cos(theta))

    for i in range(n_correlated_dimensions - 1):
        T[i, i+1] = np.sin(theta)

    for i in range(n_correlated_dimensions - 1):
        T[i+1, i] = -np.sin(theta)

    T[1, 2] = 0 # Decouple first two dimensions from rest

    # Generate data
    xi = np.random.random(n_dim)
    x = [xi]
    for i in range(n_points):
        xi = T@xi + np.random.normal(0, noise, size=(n_dim))
        x.append(xi)

    return np.stack(x)

In [13]:
ref_data = create_dataset(
    2,
    n_points=1000,
    n_correlated_dimensions=10,
    n_uncorrelated_dimensions=0,
    noise=0.2,
)

ref_dataset = TimeSeriesData(
    X=ref_data[:-1],
    y=ref_data[1:],
    lag=1,
    train_val_test_split=[0.5, 0.3, 0.2],
)

In [14]:
X, y = ref_dataset.train_data()

In [15]:
model = EigenRascuttiModel(
    bandwidth=1
)
model.fit(X, y)
model.inner_product(model)

/home/james/Repo/PhD Repo/time_series_clustering/time_series/models/eigen_rascutti.py:52: RuntimeWarning: invalid value encountered in multiply
  eval_hermitenorm(k, x)


Error in LDL factorization when computing the nonzero elements. The problem seems to be non-convex.
factor_status: 0, num_vars: 121
Error in LDL initial factorization.
ERROR: init_lin_sys_work failure


ValueError: ScsWork allocation error!

In [16]:
model = KernelRidgeRegression(
    bandwidth=1,
    kernel="rbf"
)
model.fit(X, y)
model.inner_product(model)

np.float64(5.303795177777896e+244)